# Week 3: Live Coverage Lookups with the RadioLand API

**NWR Coverage Gap Research — Summer 2026**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/W2NJL/nwr-gap-research/blob/main/week3_radioland_api.ipynb)

---

In Week 1 you explored `wx_stations.csv` — a snapshot of where NWR transmitters sit. In Week 2 you inventoried the external datasets that will tell us *who* lives near those transmitters and *what hazards* they face.

But neither of those told you something critical: **given a specific address or coordinate, what NWR signal can a receiver actually pick up there?**

That question requires a propagation model — a calculation that accounts for transmitter power, antenna height, frequency, and distance to estimate real-world signal strength at a point. This week you'll use the **RadioLand API** to run those calculations programmatically, building the core tool you'll need for the gap analysis in Weeks 4–7.

**Learning goals:**
- Understand what Server-Sent Events (SSE) are and how to consume them in Python
- Call the RadioLand API for any lat/lon and get back a ranked list of receivable NWR stations
- Interpret field strength values and establish a working coverage threshold
- Query multiple locations and start identifying which cities lack adequate coverage

---
## Part 1: Setup

In [ ]:
import requests
import json
import io
import time

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
from IPython.display import display

WX_URL     = "https://raw.githubusercontent.com/CSC-2053-Spring-26-100/lab-12-finding-the-weather-radio-gaps/main/wx_stations.csv"
CITIES_URL = "https://raw.githubusercontent.com/CSC-2053-Spring-26-100/lab-12-finding-the-weather-radio-gaps/main/us_cities.csv"

wx     = pd.read_csv(WX_URL)
wx     = wx[wx['country'] == 'USA'].copy()
wx['longitude'] = -wx['longitude'].abs()   # ensure negative for western hemisphere

cities = pd.read_csv(CITIES_URL)

print(f"Loaded {len(wx)} NWR stations and {len(cities)} cities.")

---
## Part 2: How the RadioLand API Works

### What is RadioLand?

RadioLand is an internal research tool built on top of FCC and NOAA station databases. Given a latitude and longitude, it runs an ITM (Irregular Terrain Model) propagation calculation for every NWR transmitter within range and returns a ranked list showing which stations are receivable — and how strong.

The base URL is:
```
http://52.151.197.43/search_stream
```

### Key Parameters

| Parameter | What it does |
|---|---|
| `lat`, `lon` | The receiver location you're querying |
| `broadcastBand` | `WX` for NOAA Weather Radio (NWR); others exist for FM, AM |
| `rxHeight` | Receiver antenna height above ground in meters (10 = typical home receiver) |
| `sig_strength` | Minimum field strength threshold; stations below this are excluded |
| `model` | Propagation model: `ml` uses a machine-learning-enhanced ITM calculation |
| `measurementUnit` | `metric` returns distances in km, field strength in dB(µV/m) |

### What is SSE?

Unlike a regular API that returns all results at once, this endpoint uses **Server-Sent Events (SSE)** — it streams a series of progress updates while it works, then delivers the final result when done. Each message looks like:

```
data: {"type": "progress", "percentage": 22, "message": "Calculating distances..."}

data: {"type": "complete", "percentage": 100, "data": "{...full station JSON...}"}
```

In Python, you consume it by opening the connection with `stream=True` and reading line by line until you hit the `complete` event.

---
## Part 3: The Query Function

Run the cell below — it defines `query_nwr_coverage()`, the core function you'll use throughout this notebook and in future weeks.

In [ ]:
RADIOLAND_BASE = "http://52.151.197.43/search_stream"

def query_nwr_coverage(lat, lon, rx_height=10, min_sig_strength=3, verbose=True):
    """
    Query the RadioLand API for NWR stations receivable at (lat, lon).

    Returns a DataFrame with one row per receivable station, sorted by
    field_strength descending. Returns None if the request fails.

    Parameters
    ----------
    lat, lon        : float  — receiver coordinates (lon should be negative for western US)
    rx_height       : int    — receiver antenna height above ground in meters
    min_sig_strength: int    — minimum signal threshold (3 = default per RadioLand)
    verbose         : bool   — print progress messages
    """
    params = {
        "lat": lat,
        "lon": lon,
        "search_freq": "none",
        "callsign": "none",
        "request_type": 1,
        "pi_code": "none",
        "sig_strength": min_sig_strength,
        "am_sig_strength": 2,
        "startMiles": "none",
        "miles": "null",
        "slogan": "none",
        "owner": "none",
        "format": "none",
        "wfo": "none",
        "rxHeight": rx_height,
        "mlbTeam": "none",
        "market": "none",
        "country": "none",
        "sp": "none",
        "measurementUnit": "metric",
        "locationName": "",
        "broadcastBand": "WX",
        "timeOfDay": "day",
        "model": "ml"
    }

    try:
        resp = requests.get(RADIOLAND_BASE, params=params, stream=True, timeout=90)
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f"Request failed: {e}")
        return None

    for raw_line in resp.iter_lines():
        if not raw_line:
            continue
        line = raw_line.decode('utf-8')
        if not line.startswith('data: '):
            continue

        event = json.loads(line[6:])

        if event.get('type') == 'progress' and verbose:
            print(f"  [{event['percentage']:>3}%] {event['message']}", end='\r')

        elif event.get('type') == 'complete':
            if verbose:
                print(f"  [100%] Done.                                        ")
            result = json.loads(event['data'])
            df = pd.DataFrame(result['data'])
            if df.empty:
                return df
            # The API returns positive longitudes for western-hemisphere stations — negate them
            for col in ['lon', 'transmitter_lon']:
                if col in df.columns:
                    df[col] = -df[col].abs()
            df = df.sort_values('field_strength', ascending=False).reset_index(drop=True)
            return df

    return None

print("query_nwr_coverage() is ready.")

---
## Part 4: Your First Query

Let's test the function with a known location: **Tabernacle, NJ** — a rural township in Burlington County. Run the cell and watch the progress stream in.

In [ ]:
LAT_TABERNACLE = 39.8732265
LON_TABERNACLE = -74.664345740625

print("Querying RadioLand for Tabernacle, NJ...")
tabernacle = query_nwr_coverage(LAT_TABERNACLE, LON_TABERNACLE)

print(f"\n{len(tabernacle)} receivable NWR stations found.")
tabernacle[['callsign', 'city', 'sp', 'frequency', 'distance', 'field_strength', 'wfo']].head(10)

---
## Part 5: Understanding Field Strength

The `field_strength` column is returned in **dB(µV/m)** — decibels relative to one microvolt per meter. This is the standard unit for VHF/UHF signal strength in broadcasting.

A few benchmarks for context:

| Field Strength | Interpretation |
|---|---|
| ≥ 40 dB(µV/m) | Strong — clear reception on almost any receiver |
| 20–40 dB(µV/m) | Adequate — reliable on a quality receiver, may be marginal on a cheap one |
| 10–20 dB(µV/m) | Weak — possible fringe reception, not dependable for alerts |
| < 10 dB(µV/m) | Below threshold — receiver will likely not lock on |

For this research project, we'll define **covered** as: the *best available* NWR station at a location delivers **≥ 20 dB(µV/m)**. That's a conservative but defensible threshold for reliable emergency alert reception.

You may want to revisit this threshold — a FEMA-grade standard might require higher, or the literature may cite specific FCC contour values. Note any sources you find.

### E1 — Tabernacle Coverage Profile

Use the `tabernacle` DataFrame to answer:

1. What is the strongest station (highest `field_strength`) receivable at Tabernacle? What's its callsign, city, and frequency?
2. How many stations exceed the 20 dB threshold? How many fall below 10 dB?
3. Is Tabernacle "covered" under our definition?
4. What WFO is responsible for the strongest signal? Does that match the geography?

In [ ]:
COVERAGE_THRESHOLD = 20.0   # dB(µV/m) — change this to experiment

# YOUR CODE HERE


### E2 — Field Strength vs. Distance

Plot `field_strength` (y-axis) against `distance` (x-axis) for all stations returned for Tabernacle. Add a horizontal dashed line at your coverage threshold.

Questions:
1. Is the relationship between distance and field strength perfectly linear? Why or why not?
2. Are there any outliers — stations that are far away but still strong, or close but weak? What might explain that?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

# YOUR CODE HERE
# Scatter plot of distance vs field_strength
# Add a horizontal line at COVERAGE_THRESHOLD
# Label the top 3 stations by name (callsign)

ax.set_xlabel('Distance (km)')
ax.set_ylabel('Field Strength dB(µV/m)')
ax.set_title('Tabernacle, NJ — NWR Signal Profile')
ax.axhline(COVERAGE_THRESHOLD, color='red', linestyle='--', label=f'Threshold ({COVERAGE_THRESHOLD} dB)')
ax.legend()
plt.tight_layout()
plt.show()

### E3 — Frequency Mix

NWR uses 7 frequencies (162.400–162.550 MHz in 25 kHz steps). For the Tabernacle results:

1. Which frequencies are represented?
2. How many stations are on each frequency? Do any frequencies have more coverage overlap than others?
3. Why does it matter if two strong stations share the same frequency? (Hint: look at the `Interference` column — it's empty here, but think about what it would show if it weren't.)

In [ ]:
# YOUR CODE HERE


---
## Part 6: Querying Multiple Locations

A single location query is useful for spot-checking, but our eventual goal is to classify thousands of US cities. Here you'll build a small batch query to practice the pattern.

> **Note on API etiquette:** Each call takes 15–30 seconds and runs a real propagation model. Add a short sleep between requests and limit batch sizes during development. For the large-scale city sweep in Week 5, we'll discuss a more efficient approach.

In [ ]:
def get_best_signal(lat, lon, threshold=COVERAGE_THRESHOLD):
    """
    Query RadioLand and return a summary dict for one location.
    Returns None if the request fails.
    """
    df = query_nwr_coverage(lat, lon, verbose=False)
    if df is None or df.empty:
        return {'best_callsign': None, 'best_field_strength': 0.0,
                'stations_above_threshold': 0, 'covered': False}

    best = df.iloc[0]
    return {
        'best_callsign':            best['callsign'],
        'best_field_strength':      round(best['field_strength'], 2),
        'best_distance_km':         round(best['distance'], 1),
        'best_wfo':                 best.get('wfo', None),
        'stations_above_threshold': int((df['field_strength'] >= threshold).sum()),
        'covered':                  bool(best['field_strength'] >= threshold),
    }

print("get_best_signal() is ready.")

### E4 — Query a Sample of Cities

The cell below selects a small sample of US cities and queries coverage for each. Choose the sample to include a mix of rural and urban locations, different regions, and varying population sizes.

The sampling approach is up to you — you could take the 10 smallest cities by population, or 2 cities from each of 5 states, or all cities in a specific state you care about. Justify your choice in a comment.

In [ ]:
# --- Build your sample ---
# YOUR CODE HERE — create a DataFrame called `sample` with columns:
#   city, state_id, lat, lng, population
# Keep it to 10-15 rows to avoid a very long runtime.
#
# Example (uncomment and modify):
# sample = cities[cities['state_id'] == 'WY'].nsmallest(10, 'population')

sample = None  # replace this

print(sample[['city', 'state_id', 'lat', 'lng', 'population']])

In [ ]:
# --- Run the batch query ---
# Don't modify this cell — just run it after defining `sample` above.

results = []

for _, row in sample.iterrows():
    label = f"{row['city']}, {row['state_id']}"
    print(f"Querying {label}...", end=' ')
    summary = get_best_signal(row['lat'], row['lng'])
    if summary:
        summary['city']       = row['city']
        summary['state_id']   = row['state_id']
        summary['lat']        = row['lat']
        summary['lng']        = row['lng']
        summary['population'] = row.get('population', None)
        results.append(summary)
        status = 'COVERED' if summary['covered'] else 'GAP'
        print(f"{status} — best: {summary['best_callsign']} @ {summary['best_field_strength']} dB")
    else:
        print("FAILED")
    time.sleep(2)   # be polite to the API

coverage_df = pd.DataFrame(results)
print(f"\nDone. {coverage_df['covered'].sum()} of {len(coverage_df)} cities covered.")
coverage_df

### E5 — Analyze Your Sample Results

Using `coverage_df`:

1. Which city in your sample has the weakest coverage (lowest `best_field_strength`)? What factors might explain it?
2. Is there a pattern between `population` and coverage quality? (A scatter plot might help.)
3. Are any uncovered cities surprising? Are any covered cities surprising?
4. What limitations does this 10–15 city sample have for drawing conclusions?

In [ ]:
# YOUR CODE HERE


---
## Part 7: Coverage Map

Build a Folium map showing your queried cities. Use:
- Green circles for covered cities
- Red circles for gap cities
- Popup showing city name, state, best signal, and whether it's covered

In [ ]:
m = folium.Map(location=[39.5, -98.35], zoom_start=4, tiles='CartoDB positron')

# YOUR CODE HERE
# Loop through coverage_df rows
# Add a CircleMarker for each city:
#   color = 'green' if row['covered'] else 'red'
#   popup should include city, state, best_callsign, best_field_strength

display(m)

---
## Part 8: Connecting Back to Week 2

In Week 2 you loaded (or read about) the USDA Rural-Urban Continuum Codes (RUCC). The hypothesis is: **rural areas are more likely to be NWR gaps**.

Here's a simple test of that using your sample. If you have RUCC data loaded, join it to `coverage_df` and compare average field strength across metro vs. non-metro counties. If you don't have it loaded yet, write the code as pseudocode and describe what you'd expect to find.

In [ ]:
# Optional — if you have rucc loaded from Week 2:
# rucc_joined = coverage_df.merge(
#     rucc[['State', 'County_Name', 'RUCC_2023']],
#     left_on=['state_id', 'county_name'],
#     right_on=['State', 'County_Name'],
#     how='left'
# )
# print(rucc_joined.groupby('RUCC_2023')['best_field_strength'].mean())

# YOUR CODE or pseudocode / written answer:


---
## Week 3 Reflection

Write brief answers below.

1. What does the RadioLand API give you that `wx_stations.csv` alone does not?
2. The 20 dB threshold is a starting assumption. What sources or standards would you look at to validate or revise it?
3. Two cities could both be "covered" (≥ 20 dB) but one gets 21 dB from a single station and the other gets 45 dB from three stations. Does that difference matter for emergency alerting? Why?
4. What's the biggest challenge you anticipate when scaling this query from 15 cities to 4,000+ cities in Week 5?

**Your answers:**

1. 

2. 

3. 

4. 